<a href="https://colab.research.google.com/github/renisha04/Machine-Learning-Lab/blob/main/ML_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Decision Tree   Classification and Performance

In [61]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report
)

LOAD DATASET

In [62]:
df = pd.read_csv("student_performance_balanced_1000.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)



Dataset loaded successfully!
Dataset Shape: (1000, 12)


DATA PREPROCESSING

In [63]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("%", "percent", regex=False)
)

print("\nColumn Names:")
print(df.columns.tolist())

# Remove unnecessary columns
df.drop(
    columns=["studentid", "name"],
    errors="ignore",
    inplace=True
)

# Replace infinity values
df.replace([np.inf, -np.inf], np.nan, inplace=True)


Column Names:
['studentid', 'name', 'attendancerate', 'studyhoursperweek', 'previousgrade', 'assignmentscore', 'finalgrade', 'performance', 'internalmarks', 'attendancepercent', 'studyconsistency', 'participationscore']


HANDLE INVALID VALUES

In [64]:
for column in ["attendancerate", "attendance_percent"]:
    if column in df.columns:
        df.loc[
            (df[column] < 0) | (df[column] > 100),
            column
        ] = np.nan

for column in ["previousgrade", "finalgrade"]:
    if column in df.columns:
        df.loc[
            (df[column] < 0) | (df[column] > 100),
            column
        ] = np.nan

for column in ["studyhoursperweek", "study_hours"]:
    if column in df.columns:
        df.loc[
            df[column] < 0,
            column
        ] = np.nan

REMOVE DUPLICATES

In [65]:
df.drop_duplicates(inplace=True)

print("\nDataset Shape after removing duplicates:", df.shape)



Dataset Shape after removing duplicates: (1000, 10)


CHECK FINAL GRADES

In [66]:
print("\nFinal Grade Statistics:")
print(df["finalgrade"].describe())

print("\nNumber of grades below 50:")
print((df["finalgrade"] < 50).sum())


Final Grade Statistics:
count    1000.000000
mean       63.396190
std        20.536883
min        30.140000
25%        44.870000
50%        63.135000
75%        81.247500
max        99.930000
Name: finalgrade, dtype: float64

Number of grades below 50:
334


CREATE PERFORMANCE CLASS

In [67]:
def performance_class(grade):

    if grade < 50:
        return "Fail"

    elif grade < 75:
        return "Average"

    else:
        return "Good"


df["performance"] = df["finalgrade"].apply(
    performance_class
)

print("\n===================================")
print("PERFORMANCE CLASS DISTRIBUTION")
print("===================================")

print(df["performance"].value_counts())


PERFORMANCE CLASS DISTRIBUTION
performance
Fail       334
Average    333
Good       333
Name: count, dtype: int64


CHECK WHETHER ALL 3 CLASS EXISTS

In [68]:
required_classes = ["Fail", "Average", "Good"]

missing_classes = [
    cls for cls in required_classes
    if cls not in df["performance"].unique()
]

if missing_classes:

    print("\nWARNING!")
    print("The following class/classes are missing:")
    print(missing_classes)

    print("\nYour dataset does not contain enough examples")
    print("for the model to learn these classes.")

    print("\nCurrent classes:")
    print(df["performance"].value_counts())

else:

    print("\nAll three classes are available!")
    print("Fail + Average + Good")


All three classes are available!
Fail + Average + Good


REMOVE ORIGINAL TARGET

In [69]:
df.drop(
    columns=["finalgrade"],
    inplace=True
)

SEPARATE FEATURES AND TARGET

In [70]:
X = df.drop(
    columns=["performance"]
)

y = df["performance"]

ENCODING

In [71]:
X = pd.get_dummies(
     X,
    drop_first=True,
    dtype=int
)


TRAIN-TEST SPLIT

In [72]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("\n===================================")
print("TRAIN TEST SPLIT")
print("===================================")

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)



TRAIN TEST SPLIT
Training data: (800, 8)
Testing data : (200, 8)


HANDLE MISSING VALUES

In [73]:
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)


TRAIN DECISION TREE

In [74]:
dt_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42
)

dt_model.fit(
    X_train,
    y_train
)

print("\n===================================")
print("DECISION TREE")
print("===================================")

print("Decision Tree model trained successfully!")


DECISION TREE
Decision Tree model trained successfully!


PREDICTION

In [75]:
y_pred = dt_model.predict(X_test)

CONFUSION MATRIX

In [76]:
labels = ["Fail", "Average", "Good"]

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

print("\n===================================")
print("CONFUSION MATRIX")
print("===================================")

print("             Predicted")
print("          Fail Average Good")
print("--------------------------------")

for i, label in enumerate(labels):
    print(
        f"{label:<8}",
        cm[i]
    )




CONFUSION MATRIX
             Predicted
          Fail Average Good
--------------------------------
Fail     [64  3  0]
Average  [ 0 66  1]
Good     [ 0  0 66]


ACCURACY

In [77]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\n===================================")
print("ACCURACY")
print("===================================")

print("Accuracy:", accuracy)
print(
    "Accuracy Percentage:",
    round(accuracy * 100, 2),
    "%"
)



ACCURACY
Accuracy: 0.98
Accuracy Percentage: 98.0 %


CLASSIFICATION REPORT

In [78]:
print("\n===================================")
print("CLASSIFICATION REPORT")
print("===================================")

print(
    classification_report(
        y_test,
        y_pred,
        labels=labels,
        zero_division=0
    )
)


CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Fail       1.00      0.96      0.98        67
     Average       0.96      0.99      0.97        67
        Good       0.99      1.00      0.99        66

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200



SAMPLE PREDICTIONS

In [79]:
print("\n===================================")
print("SAMPLE PREDICTIONS")
print("===================================")

sample_output = pd.DataFrame({
    "Actual": y_test.iloc[:20].values,
    "Predicted": y_pred[:20]
})

print(sample_output)



SAMPLE PREDICTIONS
     Actual Predicted
0   Average   Average
1      Good      Good
2      Fail   Average
3      Fail      Fail
4      Good      Good
5      Fail      Fail
6      Good      Good
7   Average   Average
8   Average   Average
9   Average   Average
10  Average   Average
11     Good      Good
12  Average   Average
13     Good      Good
14  Average   Average
15     Fail      Fail
16  Average   Average
17     Fail      Fail
18     Good      Good
19     Fail      Fail


PREDICTION COUNT

In [80]:
print("\n===================================")
print("PREDICTED CLASS DISTRIBUTION")
print("===================================")

print(
    pd.Series(y_pred).value_counts()
)


PREDICTED CLASS DISTRIBUTION
Average    69
Good       67
Fail       64
Name: count, dtype: int64
